# 10 — Exportación del dataframe final a Excel

Toma el corpus clasificado (`scam_us_FINAL_classified.csv`) y genera un Excel multi-hoja con formato profesional para entrega al tutor.

## Hojas del Excel generado

1. **Resumen** — totales por categoría, confianza media, métricas globales.
2. **Corpus completo** — todas las filas, columnas seleccionadas.
3. **Solo relevantes** — sin los "not_related", listo para análisis.
4. **Tópicos BERTopic** — distribución de tópicos descubiertos.
5. **Ejemplos por categoría** — 5 ejemplos representativos por cada una de las 14 categorías.

## Requisitos

- `openpyxl` instalado (viene con pandas pero por si acaso).
- El CSV `scam_us_FINAL_classified.csv` en `../data/raw/`.


In [ ]:
# Verificación rápida
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# Instalar openpyxl si no está (lo necesitamos para .xlsx con formato)
try:
    import openpyxl
    print(f"✓ openpyxl {openpyxl.__version__}")
except ImportError:
    print("Instalando openpyxl...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])
    import openpyxl
    print(f"✓ openpyxl {openpyxl.__version__}")

print(f"✓ pandas {pd.__version__}")


## Carga del corpus clasificado

In [ ]:
INPUT_PATH = "../data/raw/scam_us_FINAL_classified.csv"
df = pd.read_csv(INPUT_PATH)
print(f"Tweets cargados: {len(df)}")
print(f"Columnas disponibles: {list(df.columns)}")


## Hoja 1 — Resumen estadístico

In [ ]:
# Métricas globales
total_tweets = len(df)
total_relevant = int(df["is_relevant"].sum())
pct_relevant = total_relevant / total_tweets * 100
conf_media = df["confidence_score"].mean()
conf_mediana = df["confidence_score"].median()
n_topics = df["bertopic_id"].nunique() - (1 if -1 in df["bertopic_id"].values else 0)

resumen_global = pd.DataFrame({
    "Métrica": [
        "Total de tweets clasificados",
        "Tweets relevantes (fraude identificado)",
        "Tweets no relacionados con fraude",
        "% tweets relevantes",
        "Confianza media",
        "Confianza mediana",
        "Tweets con alta confianza (≥0.7)",
        "Tweets con confianza baja (<0.4)",
        "Tópicos BERTopic identificados",
        "Outliers BERTopic",
    ],
    "Valor": [
        total_tweets,
        total_relevant,
        total_tweets - total_relevant,
        f"{pct_relevant:.1f}%",
        f"{conf_media:.3f}",
        f"{conf_mediana:.3f}",
        int((df["confidence_score"] >= 0.7).sum()),
        int((df["confidence_score"] < 0.4).sum()),
        n_topics,
        int((df["bertopic_id"] == -1).sum()),
    ],
})

# Distribución por categoría
dist_categorias = (
    df.groupby("predicted_category")
    .agg(
        n_tweets=("tweet_id", "count"),
        confianza_media=("confidence_score", "mean"),
        confianza_min=("confidence_score", "min"),
        confianza_max=("confidence_score", "max"),
    )
    .sort_values("n_tweets", ascending=False)
    .reset_index()
)
dist_categorias["porcentaje"] = (dist_categorias["n_tweets"] / total_tweets * 100).round(2)
dist_categorias = dist_categorias[["predicted_category", "n_tweets", "porcentaje",
                                    "confianza_media", "confianza_min", "confianza_max"]]
dist_categorias["confianza_media"] = dist_categorias["confianza_media"].round(3)
dist_categorias["confianza_min"] = dist_categorias["confianza_min"].round(3)
dist_categorias["confianza_max"] = dist_categorias["confianza_max"].round(3)

print("=== RESUMEN GLOBAL ===")
print(resumen_global.to_string(index=False))
print()
print("=== DISTRIBUCIÓN POR CATEGORÍA ===")
print(dist_categorias.to_string(index=False))


## Hoja 2 — Corpus completo (columnas seleccionadas)

In [ ]:
# Seleccionamos las columnas más relevantes para entrega
columnas_corpus = [
    "tweet_id", "created_at", "text",
    "username", "user_location",
    "user_followers", "retweet_count", "like_count",
    "query_labels",
    "bertopic_id", "bertopic_keywords",
    "predicted_label", "predicted_category", "confidence_score", "is_relevant",
]
columnas_corpus = [c for c in columnas_corpus if c in df.columns]
corpus_export = df[columnas_corpus].copy()
print(f"Corpus completo: {len(corpus_export)} filas × {len(columnas_corpus)} columnas")
print(f"Columnas: {columnas_corpus}")


## Hoja 3 — Solo tweets relevantes (sin 'not_related')

In [ ]:
corpus_relevantes = corpus_export[corpus_export["is_relevant"]].copy()
print(f"Tweets relevantes: {len(corpus_relevantes)} ({len(corpus_relevantes)/len(corpus_export)*100:.1f}% del total)")


## Hoja 4 — Tópicos BERTopic

In [ ]:
topicos_resumen = (
    df.groupby(["bertopic_id", "bertopic_keywords"])
    .size()
    .reset_index(name="n_tweets")
    .sort_values("n_tweets", ascending=False)
)
print(f"Tópicos identificados (incluyendo outliers): {len(topicos_resumen)}")
print()
print(topicos_resumen.head(20).to_string(index=False))


## Hoja 5 — Ejemplos representativos por categoría

In [ ]:
# Para cada categoría, los 5 tweets de mayor confianza
ejemplos_list = []
for cat in df["predicted_category"].unique():
    if pd.isna(cat):
        continue
    subset = df[df["predicted_category"] == cat].nlargest(5, "confidence_score")
    for _, row in subset.iterrows():
        ejemplos_list.append({
            "categoría": cat,
            "confianza": round(row["confidence_score"], 3),
            "tweet": str(row["text"])[:300] if pd.notna(row["text"]) else "",
            "usuario": row.get("username", ""),
            "ubicación": row.get("user_location", ""),
            "tópico_bertopic": row.get("bertopic_id", ""),
        })

ejemplos_df = pd.DataFrame(ejemplos_list)
print(f"Ejemplos preparados: {len(ejemplos_df)}")
print(ejemplos_df.head(10).to_string(index=False))


## Guardado del Excel con formato

In [ ]:
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

OUTPUT_PATH = "../data/processed/scam_us_FINAL_dataframe.xlsx"
Path("../data/processed").mkdir(parents=True, exist_ok=True)

# Escribir el Excel con varias hojas
with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    resumen_global.to_excel(writer, sheet_name="Resumen", index=False, startrow=0)
    # Espacio entre tablas
    dist_categorias.to_excel(writer, sheet_name="Resumen", index=False, startrow=len(resumen_global) + 3)
    corpus_export.to_excel(writer, sheet_name="Corpus completo", index=False)
    corpus_relevantes.to_excel(writer, sheet_name="Solo relevantes", index=False)
    topicos_resumen.to_excel(writer, sheet_name="Tópicos BERTopic", index=False)
    ejemplos_df.to_excel(writer, sheet_name="Ejemplos por categoría", index=False)

# Ahora aplicamos formato
from openpyxl import load_workbook
wb = load_workbook(OUTPUT_PATH)

# Colores por categoría para resaltar
category_colors = {
    "investment_crypto":  "FFE4B5",
    "romance":            "FFB6C1",
    "phishing_identity":  "ADD8E6",
    "gov_impersonation":  "98FB98",
    "bank_wire":          "DDA0DD",
    "payment_app":        "FFA07A",
    "ponzi_pyramid":      "F0E68C",
    "tech_support":       "B0C4DE",
    "employment":         "FFDAB9",
    "charity":            "E6E6FA",
    "insurance":          "F5DEB3",
    "corporate":          "D3D3D3",
    "tax":                "FFCCCC",
    "not_related":        "DCDCDC",
}

header_font = Font(bold=True, color="FFFFFF", size=11)
header_fill = PatternFill(start_color="305496", end_color="305496", fill_type="solid")
center_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
left_align = Alignment(horizontal="left", vertical="top", wrap_text=True)

thin_border = Border(
    left=Side(style='thin', color='B0B0B0'),
    right=Side(style='thin', color='B0B0B0'),
    top=Side(style='thin', color='B0B0B0'),
    bottom=Side(style='thin', color='B0B0B0'),
)

for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]

    # Formato de cabeceras (primera fila con contenido)
    for row in ws.iter_rows(min_row=1, max_row=1):
        for cell in row:
            if cell.value is not None:
                cell.font = header_font
                cell.fill = header_fill
                cell.alignment = center_align
                cell.border = thin_border

    # Para "Resumen", también formatear la segunda cabecera (de la tabla de categorías)
    if sheet_name == "Resumen":
        second_header_row = len(resumen_global) + 4
        for cell in ws[second_header_row]:
            if cell.value is not None:
                cell.font = header_font
                cell.fill = header_fill
                cell.alignment = center_align
                cell.border = thin_border

    # Anchura automática de columnas (con tope)
    for col_idx, col in enumerate(ws.columns, start=1):
        max_len = 0
        for cell in col:
            try:
                if cell.value:
                    max_len = max(max_len, min(len(str(cell.value)), 80))
            except:
                pass
        ws.column_dimensions[get_column_letter(col_idx)].width = min(max(max_len + 2, 12), 60)

    # Color por categoría para las hojas que tienen 'predicted_category' o 'categoría'
    cat_col_idx = None
    for col_idx, cell in enumerate(ws[1], start=1):
        if cell.value in ("predicted_category", "categoría"):
            cat_col_idx = col_idx
            break

    if cat_col_idx:
        for row in ws.iter_rows(min_row=2):
            cat_value = row[cat_col_idx - 1].value
            if cat_value in category_colors:
                fill = PatternFill(start_color=category_colors[cat_value],
                                   end_color=category_colors[cat_value],
                                   fill_type="solid")
                for cell in row:
                    cell.fill = fill

    # Autofiltro en hojas con datos tabulares
    if sheet_name not in ("Resumen",) and ws.max_row > 1:
        ws.auto_filter.ref = ws.dimensions

    # Congelar primera fila
    ws.freeze_panes = "A2"

wb.save(OUTPUT_PATH)
print(f"✓ Excel guardado: {OUTPUT_PATH}")
print(f"  Hojas: {wb.sheetnames}")
print(f"  Tamaño aproximado: {Path(OUTPUT_PATH).stat().st_size / 1024:.0f} KB")


## Verificación final

In [ ]:
import os
print("=== ARCHIVO GENERADO ===")
print(f"Ruta:    {OUTPUT_PATH}")
print(f"Tamaño:  {os.path.getsize(OUTPUT_PATH) / 1024:.1f} KB")
print()
print("Para abrirlo:")
print("  - Doble click en Finder")
print("  - Se abre con Excel / Numbers / Google Sheets")
print()
print("📌 Listo para entregar al tutor.")
